<a href="https://colab.research.google.com/github/grbagwe/AIOT-jetson/blob/main/PTQ_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, time, math, tempfile, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

In [2]:
torch.manual_seed(0); random.seed(0)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")  # quantization targets CPU

print(device)

cpu


In [3]:

# 1) Data
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="/tmp/mnist", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="/tmp/mnist", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)



100%|██████████| 9.91M/9.91M [00:00<00:00, 128MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 15.9MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 117MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.57MB/s]


In [4]:
# 2) Model (fusable blocks)
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, stride=1, padding=1)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(16, 32, 3, stride=1, padding=1)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2,2)
        self.fc1   = nn.Linear(32*7*7, 128)
        self.relu3 = nn.ReLU(inplace=True)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.reshape(x.size(0), -1) # Changed view to reshape
        x = self.relu3(self.fc1(x))
        return self.fc2(x)

In [5]:

# 3) Train FP32 baseline
def train_epoch(m, opt, loader):
    m.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = m(x)
        loss = F.cross_entropy(logits, y)
        loss.backward(); opt.step()
        loss_sum += loss.item()*x.size(0)
        correct += (logits.argmax(1)==y).sum().item()
        total += x.size(0)
    return loss_sum/total, correct/total

def evaluate(m, loader):
    m.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            logits = m(x)
            loss_sum += F.cross_entropy(logits, y, reduction='sum').item()
            correct += (logits.argmax(1)==y).sum().item()
            total += x.size(0)
    return loss_sum/total, correct/total


In [6]:

model_fp32 = SmallCNN().to(device)
opt = torch.optim.Adam(model_fp32.parameters(), lr=1e-3)

for epoch in range(2):  # fast for demo; increase to 3–5 for higher accuracy
    tr_loss, tr_acc = train_epoch(model_fp32, opt, train_loader)
    print(f"epoch {epoch+1}: train_loss={tr_loss:.4f} train_acc={tr_acc*100:.2f}%")

te_loss, te_acc = evaluate(model_fp32, test_loader)
print(f"FP32 test_acc={te_acc*100:.2f}%")



epoch 1: train_loss=0.2949 train_acc=91.66%
epoch 2: train_loss=0.0696 train_acc=97.81%
FP32 test_acc=98.36%


In [7]:
# 4) Size and latency helpers
def save_size_mb(state_dict, path):
    torch.save(state_dict, path)
    return os.path.getsize(path)/1e6

In [8]:

def benchmark_latency(m, batch_size=1, iters=200, warmup=20):
    m.eval()
    x = torch.randn(batch_size,1,28,28, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = m(x)
        t0 = time.time()
        for _ in range(iters):
            _ = m(x)
        t1 = time.time()
    avg_ms = (t1-t0)/iters*1000
    return avg_ms

In [9]:
tmpdir = tempfile.mkdtemp()
fp32_size_mb = save_size_mb(model_fp32.state_dict(), os.path.join(tmpdir,"mnist_fp32.pt"))
fp32_latency_ms = benchmark_latency(model_fp32, batch_size=128)

print(f"FP32 size={fp32_size_mb:.2f} MB, latency(128x1x28x28)={fp32_latency_ms:.2f} ms")

# 5) PTQ (static) with FX Graph Mode
model_fp32.eval()

# Fuse by rewriting model as TorchScript? Not needed with FX; set backend + qconfig.
backend = "fbgemm"
torch.backends.quantized.engine = backend
qconfig = get_default_qconfig(backend)

example_inputs = torch.randn(1,1,28,28)
prepared = prepare_fx(model_fp32, {"": qconfig}, example_inputs=example_inputs)

FP32 size=0.83 MB, latency(128x1x28x28)=30.73 ms


/tmp/ipython-input-1155899231.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared = prepare_fx(model_fp32, {"": qconfig}, example_inputs=example_inputs)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass 

In [10]:
tmpdir = tempfile.mkdtemp()
fp32_size_mb = save_size_mb(model_fp32.state_dict(), os.path.join(tmpdir,"mnist_fp32.pt"))
fp32_latency_ms = benchmark_latency(model_fp32, batch_size=1)

print(f"FP32 size={fp32_size_mb:.2f} MB, latency(1x28x28)={fp32_latency_ms:.2f} ms")

# 5) PTQ (static) with FX Graph Mode
model_fp32.eval()

# Fuse by rewriting model as TorchScript? Not needed with FX; set backend + qconfig.
backend = "fbgemm"
torch.backends.quantized.engine = backend
qconfig = get_default_qconfig(backend)

example_inputs = torch.randn(1,1,28,28)
prepared = prepare_fx(model_fp32, {"": qconfig}, example_inputs=example_inputs)

FP32 size=0.83 MB, latency(1x28x28)=0.73 ms


/tmp/ipython-input-3744289064.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared = prepare_fx(model_fp32, {"": qconfig}, example_inputs=example_inputs)


In [11]:
# Calibration on a few batches
prepared.eval()
with torch.no_grad():
    for i,(x,_) in enumerate(train_loader):
        _ = prepared(x.to(device))
        if i >= 100:  # ~128*101 ≈ 12.9k samples; adjust for speed
            break

quantized_model = convert_fx(prepared).to(device)


/tmp/ipython-input-3597674952.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = convert_fx(prepared).to(device)


In [12]:

# 6) Evaluate quantized model
q_loss, q_acc = evaluate(quantized_model, test_loader)
q_size_mb = save_size_mb(quantized_model.state_dict(), os.path.join(tmpdir,"mnist_int8.pt"))
q_latency_ms = benchmark_latency(quantized_model, batch_size=1)


In [13]:

# 7) Cosine similarity of logits (sanity)
def cosine_logits(m1, m2, samples=2000):
    m1.eval(); m2.eval()
    sims = []
    n = 0
    with torch.no_grad():
        for x,_ in test_loader:
            l1 = m1(x)
            l2 = m2(x)
            num = (l1*l2).sum(dim=1)
            den = l1.norm(dim=1)*l2.norm(dim=1) + 1e-8
            sims.append((num/den).cpu())
            n += x.size(0)
            if n >= samples: break
    return torch.cat(sims)[:samples].mean().item()

cos_sim = cosine_logits(model_fp32, quantized_model, samples=2000)

# 8) Report
print("\n=== Results ===")
print(f"Accuracy FP32     : {te_acc*100:.2f}%")
print(f"Accuracy INT8 PTQ : {q_acc*100:.2f}%")
print(f"Δ Accuracy        : {(q_acc-te_acc)*100:.2f} pp")
print(f"Size FP32         : {fp32_size_mb:.2f} MB")
print(f"Size INT8         : {q_size_mb:.2f} MB")
print(f"Reduction         : {fp32_size_mb/q_size_mb:.2f}×")
print(f"Latency FP32      : {fp32_latency_ms:.2f} ms")
print(f"Latency INT8      : {q_latency_ms:.2f} ms")
print(f"Speedup           : {fp32_latency_ms/q_latency_ms:.2f}×")
print(f"Cosine(logits)    : {cos_sim:.4f}")

# 9) Optional: per-batch latency distribution (quick)
def latency_distribution(m, runs=50):
    m.eval()
    xs = [torch.randn(1,1,28,28) for _ in range(runs)]
    times = []
    with torch.no_grad():
        for x in xs:
            t0 = time.time(); _ = m(x); t1 = time.time()
            times.append((t1-t0)*1000)
    return sum(times)/len(times), min(times), max(times)

mean_fp32, min_fp32, max_fp32 = latency_distribution(model_fp32)
mean_int8, min_int8, max_int8 = latency_distribution(quantized_model)
print("\nLatency dist (ms) [mean, min, max]")
print(f"FP32 : {mean_fp32:.2f}, {min_fp32:.2f}, {max_fp32:.2f}")
print(f"INT8 : {mean_int8:.2f}, {min_int8:.2f}, {max_int8:.2f}")


=== Results ===
Accuracy FP32     : 98.36%
Accuracy INT8 PTQ : 98.45%
Δ Accuracy        : 0.09 pp
Size FP32         : 0.83 MB
Size INT8         : 0.22 MB
Reduction         : 3.80×
Latency FP32      : 0.73 ms
Latency INT8      : 0.64 ms
Speedup           : 1.15×
Cosine(logits)    : 0.9999

Latency dist (ms) [mean, min, max]
FP32 : 0.69, 0.62, 1.17
INT8 : 0.86, 0.79, 1.30
